# Universal Approximation Theorem from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/universal_approximation_theorem.ipynb)

A one-hidden-layer network with a non-polynomial activation can approximate any
continuous function on a bounded region to any accuracy, given enough width
(Cybenko 1989; Hornik, Stinchcombe & White 1989). This notebook builds such a
network from scratch, sweeps its width, watches the error collapse, and shows
**why** it works: each hidden unit is one bump (or one kink), and a sum of bumps
tiles any curve.

Accompanies the blog post:
[Universal Approximation Theorem from Scratch](https://sesen.ai/blog/universal-approximation-theorem-from-scratch).

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

BLUE, ORANGE, GREY = "#2563eb", "#f97316", "#64748b"
plt.rcParams.update({"font.size": 12, "axes.grid": True, "grid.alpha": 0.25})

## The target and the network

We fit a deliberately wiggly target (three stacked sine waves) with a single
hidden layer. Written out, the network computes

$$f(x) = \sum_{i=1}^{N} v_i\, \sigma(w_i x + b_i) + c$$

where `N` is the width and `sigma` is the activation. That sum is the whole model.

In [ ]:
def target(x):
    '''A continuous but wiggly target: three stacked sine waves on [-1, 1].'''
    return (np.sin(2 * np.pi * x)
            + 0.5 * np.sin(6 * np.pi * x)
            + 0.25 * np.sin(10 * np.pi * x))


class OneHiddenLayer(nn.Module):
    '''One hidden layer: f(x) = W2 . act(W1 . x + b1) + b2, `width` units.'''

    def __init__(self, width, act="relu"):
        super().__init__()
        self.fc1 = nn.Linear(1, width)
        self.fc2 = nn.Linear(width, 1)
        self.act = torch.relu if act == "relu" else torch.tanh

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

The `fit` routine trains each network with L-BFGS and keeps the best of several
random restarts. Restarts separate the network's approximation **capacity** from
the luck of one optimiser run, so the error reflects what the architecture can
represent rather than where a single gradient descent landed.

In [ ]:
def fit(width, x, y, act="relu", restarts=6, rounds=15):
    '''Fit the width-`width` net; keep the best of several random restarts.'''
    xt = torch.tensor(x, dtype=torch.float32).view(-1, 1)
    yt = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    loss_fn = nn.MSELoss()
    best_net, best_mse = None, float("inf")
    for s in range(restarts):
        torch.manual_seed(1000 * s + width)
        net = OneHiddenLayer(width, act)
        opt = torch.optim.LBFGS(net.parameters(), lr=0.8, max_iter=50,
                                line_search_fn="strong_wolfe")

        def closure():
            opt.zero_grad()
            loss = loss_fn(net(xt), yt)
            loss.backward()
            return loss

        for _ in range(rounds):
            opt.step(closure)
        with torch.no_grad():
            mse = loss_fn(net(xt), yt).item()
        if mse < best_mse:
            best_net, best_mse = net, mse
    return best_net, best_mse


def predict(net, x):
    with torch.no_grad():
        return net(torch.tensor(x, dtype=torch.float32).view(-1, 1)).numpy().ravel()

## Quick win: wider layer, closer fit

In [ ]:
x = np.linspace(-1, 1, 400)
y = target(x)

for w in [1, 4, 16, 256]:
    _, mse = fit(w, x, y)
    print(f"width {w:>3}:  MSE {mse:.2e}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, w in zip(axes.ravel(), [1, 4, 16, 256]):
    net, mse = fit(w, x, y)
    ax.plot(x, y, color=GREY, lw=3, alpha=0.55, label="target")
    ax.plot(x, predict(net, x), color=BLUE, lw=2, label="net")
    ax.set_title(f"width {w}   (MSE {mse:.1e})")
    ax.set_ylim(-2, 2)
    ax.legend(loc="upper right", fontsize=9)
fig.suptitle("More hidden units, closer fit")
fig.tight_layout()
plt.show()

## Why it works: soft steps and bumps

A sigmoid is a soft step; the weight controls its sharpness. Subtract two
opposing steps and you get a localised **bump**, the atom of the construction.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def bump(xg, a, b, s=120.0):
    '''A localised bump on [a, b] from two opposing soft steps.'''
    return sigmoid(s * (xg - a)) - sigmoid(s * (xg - b))


xg = np.linspace(-1, 1, 800)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for steep in [3, 10, 60]:
    axes[0].plot(xg, sigmoid(steep * xg), lw=2, label=f"steepness {steep}")
axes[0].set_title("A sigmoid is a soft step")
axes[0].legend()
sa, sb = sigmoid(30 * (xg + 0.2)), sigmoid(30 * (xg - 0.2))
axes[1].plot(xg, sa, "--", color=BLUE, label="step up at a")
axes[1].plot(xg, sb, "--", color=ORANGE, label="step up at b")
axes[1].plot(xg, sa - sb, color="#0f766e", lw=3, label="difference = bump")
axes[1].set_title("Two opposing steps make a bump")
axes[1].legend()
plt.show()

### Bumps tile any curve

Chop the domain into strips; on each strip add a bump scaled to the target's
value there. The sum traces the curve, with no training at all: a network that
approximates the target provably **exists**, and here it is.

In [ ]:
K = 40
edges = np.linspace(-1, 1, K + 1)
centres = 0.5 * (edges[:-1] + edges[1:])

approx = np.zeros_like(xg)
fig, ax = plt.subplots(figsize=(9, 5))
for c, a, b in zip(centres, edges[:-1], edges[1:]):
    piece = target(c) * bump(xg, a, b)
    approx += piece
    ax.plot(xg, piece, color=ORANGE, lw=0.8, alpha=0.5)
ax.plot(xg, approx, color=BLUE, lw=2.4, label=f"sum of {K} bumps")
ax.plot(xg, target(xg), color=GREY, lw=3, alpha=0.5, label="target")
ax.set_title("Each hidden unit is one bump; their sum tiles the curve")
ax.set_ylim(-2.2, 2.2)
ax.legend(loc="upper right")
plt.show()

print(f"MSE of hand-built {K}-bump network: {np.mean((approx - target(xg))**2):.2e}")

### ReLU: a piecewise-linear tiling

The networks we trained use ReLU. Each ReLU unit is off until its input crosses
a threshold, then switches on as a ramp. A ReLU network is therefore
piecewise-linear: straight segments joined at kinks, one kink per active unit.

In [ ]:
w = 12
net, mse = fit(w, x, y, act="relu")
W1 = net.fc1.weight.detach().numpy().ravel()
b1 = net.fc1.bias.detach().numpy().ravel()
kinks = -b1 / np.where(np.abs(W1) < 1e-8, 1e-8, W1)
kinks = kinks[(kinks > -1) & (kinks < 1)]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, y, color=GREY, lw=3, alpha=0.5, label="target")
ax.plot(x, predict(net, x), color=BLUE, lw=2.4, label=f"{w}-unit ReLU net")
for k in kinks:
    ax.axvline(k, color=ORANGE, lw=0.9, alpha=0.6)
ax.plot([], [], color=ORANGE, lw=1, label="ReLU kinks")
ax.set_title("A ReLU network is a piecewise-linear function")
ax.set_ylim(-2, 2)
ax.legend(loc="upper right")
plt.show()
print(f"{len(kinks)} kinks inside [-1, 1], MSE {mse:.2e}")

## Error versus width

Fit the target at every width from 1 to 256 and plot the error on log-log axes.
ReLU error falls roughly like `1/width`; smooth tanh units are even more
efficient on this smooth target. There is no floor the error refuses to cross:
that is what "arbitrary accuracy" looks like.

In [ ]:
widths = [1, 2, 4, 8, 16, 32, 64, 128, 256]
relu_mse = [fit(w, x, y, act="relu")[1] for w in widths]
# wide tanh nets are harder to optimise, so give them more restarts
tanh_mse = [fit(w, x, y, act="tanh", restarts=10)[1] for w in widths]

fig, ax = plt.subplots(figsize=(8, 5))
wr = np.array(widths)
ax.loglog(wr, relu_mse, "o-", color=BLUE, lw=2, ms=7, label="ReLU units")
ax.loglog(wr, tanh_mse, "s--", color=ORANGE, lw=2, ms=6, label="tanh units")
ax.loglog(wr, relu_mse[0] * (wr / wr[0]) ** -1.0, ":", color=GREY,
          label="1/width reference")
ax.set_xlabel("hidden units (width)")
ax.set_ylabel("approximation error (MSE)")
ax.set_title("Approximation error collapses as width grows")
ax.legend()
ax.grid(True, which="both", alpha=0.25)
plt.show()

**Existence is not learnability.** The tanh curve may wobble upward at the widest
size. A wider network can only represent the target better, so this is the
optimiser failing to find weights that provably exist, not a capacity limit.
The theorem guarantees a good network is out there; gradient descent finding it
is a separate promise.

## Beyond one dimension

The same recipe fits a two-dimensional surface. Swap the input layer for two
inputs and fit a `sin(3*x0) * cos(3*x1)` checkerboard.

In [ ]:
def surface(X):
    return (np.sin(3 * X[:, 0]) * np.cos(3 * X[:, 1])).reshape(-1, 1)


class OneHidden2D(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.fc1 = nn.Linear(2, width)
        self.fc2 = nn.Linear(width, 1)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))


rng = np.random.default_rng(0)
Xtr = rng.uniform(-2, 2, size=(3000, 2)).astype(np.float32)
ytr = surface(Xtr).astype(np.float32)
Xt, yt = torch.tensor(Xtr), torch.tensor(ytr)

gg = np.linspace(-2, 2, 120)
G0, G1 = np.meshgrid(gg, gg)
grid = np.column_stack([G0.ravel(), G1.ravel()]).astype(np.float32)
true = surface(grid).reshape(G0.shape)

preds = {}
for width in [8, 256]:
    torch.manual_seed(0)
    net = OneHidden2D(width)
    opt = torch.optim.Adam(net.parameters(), lr=0.01)
    lossf = nn.MSELoss()
    for _ in range(3000):
        opt.zero_grad()
        loss = lossf(net(Xt), yt)
        loss.backward()
        opt.step()
    with torch.no_grad():
        preds[width] = (net(torch.tensor(grid)).numpy().reshape(G0.shape),
                        loss.item())

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
panels = [("target", true), (f"width 8 (MSE {preds[8][1]:.1e})", preds[8][0]),
          (f"width 256 (MSE {preds[256][1]:.1e})", preds[256][0])]
for ax, (title, Z) in zip(axes, panels):
    im = ax.imshow(Z, extent=[-2, 2, -2, 2], origin="lower",
                   cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_title(title)
    ax.grid(False)
plt.show()

## Exercises

1. **Change the target.** Swap in a sawtooth, a step, or `|x|`. How many ReLU
   units does each need before the fit looks right? Which is hardest?
2. **Sharper bumps.** In the hand-built bump network, raise the steepness `s` and
   the number of bumps `K`. How low can you push the MSE with no training?
3. **Existence versus learnability.** Replace L-BFGS with plain SGD or Adam in
   `fit`. Does the error-versus-width curve stay as clean? Where does the
   optimiser start to struggle?
4. **Depth versus width.** Add a second hidden layer of the same width. For a
   fixed parameter budget, does deeper beat wider on a high-frequency target?
5. **Extrapolation.** Fit on `[-1, 1]` but evaluate on `[-2, 2]`. The theorem
   only promises accuracy on the region you trained on. What happens outside it?